# Aggregate y_pred_test Data Across Universes

This notebook aggregates test predictions from all universes after a multiverse analysis run.

## Import Required Libraries

In [ ]:
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

## Configuration

Set the run ID to aggregate predictions for.

In [ ]:
# Set the run ID you want to aggregate
run_id = 0

# Base path to the output directory
base_path = Path("output/runs") / str(run_id) / "exports"
print(f"Looking for predictions in: {base_path}")

## Find All y_pred_test Files

Search for all y_pred_test.parquet files across different universes.

In [ ]:
# Find all y_pred_test.parquet files
pred_files = list(base_path.glob("*/y_pred_test.parquet"))

print(f"Found {len(pred_files)} prediction files")
print("\nFirst few files:")
for f in pred_files[:5]:
    print(f"  - {f}")

## Aggregate Predictions from All Universes

Load each parquet file and add the universe_id to the data.

In [ ]:
# Aggregate all predictions
all_predictions = []

for pred_file in pred_files:
    # Extract universe_id from the directory name
    universe_id = pred_file.parent.name

    # Read the parquet file
    df = pd.read_parquet(pred_file)

    # Add universe_id column
    df['universe_id'] = universe_id

    # Append to list
    all_predictions.append(df)

# Concatenate all dataframes
if all_predictions:
    # First create long format
    long_df = pd.concat(all_predictions, ignore_index=True)
    print(f"Loaded {len(long_df):,} predictions from {len(all_predictions)} universes")

    # Identify ID column and prediction columns
    id_col = [c for c in long_df.columns if 'id' in c.lower() and c != 'universe_id'][0] if any('id' in c.lower() and c != 'universe_id' for c in long_df.columns) else long_df.columns[0]

    # Separate y_pred and y_pred_proba columns
    y_pred_cols = [c for c in long_df.columns if c.startswith('y_pred') and 'proba' not in c]
    y_pred_proba_cols = [c for c in long_df.columns if 'proba' in c]

    print(f"\nFound {len(y_pred_cols)} y_pred columns and {len(y_pred_proba_cols)} y_pred_proba columns")

    # Check for duplicates within universes
    duplicates = long_df.groupby(['universe_id', id_col]).size()
    if duplicates.max() > 1:
        print("\nWarning: Found duplicate IDs within universes. Taking first occurrence.")
        print(f"Max duplicates per ID/universe: {duplicates.max()}")
        # Keep only the first occurrence of each ID within each universe
        long_df = long_df.drop_duplicates(subset=['universe_id', id_col], keep='first')
        print(f"After removing duplicates: {len(long_df):,} predictions")

    # Create separate dataframes for y_pred and y_pred_proba
    y_pred_df = None
    y_pred_proba_df = None

    if y_pred_cols:
        if len(y_pred_cols) == 1:
            # Single prediction column - simple pivot
            y_pred_df = long_df.pivot(index=id_col, columns='universe_id', values=y_pred_cols[0])
            y_pred_df.columns = [f'pred_{col}' for col in y_pred_df.columns]
            y_pred_df = y_pred_df.reset_index()
        else:
            # Multiple prediction columns - pivot each separately
            y_pred_df = long_df.pivot(index=id_col, columns='universe_id', values=y_pred_cols)
            y_pred_df.columns = ['_'.join(map(str, col)) for col in y_pred_df.columns]
            y_pred_df = y_pred_df.reset_index()

        print(f"\ny_pred shape: {y_pred_df.shape}")
        print(f"Columns: {len(y_pred_df.columns)} (1 ID + {len(y_pred_df.columns)-1} prediction columns)")

    if y_pred_proba_cols:
        if len(y_pred_proba_cols) == 1:
            # Single prediction column - simple pivot
            y_pred_proba_df = long_df.pivot(index=id_col, columns='universe_id', values=y_pred_proba_cols[0])
            y_pred_proba_df.columns = [f'pred_{col}' for col in y_pred_proba_df.columns]
            y_pred_proba_df = y_pred_proba_df.reset_index()
        else:
            # Multiple prediction columns - pivot each separately
            y_pred_proba_df = long_df.pivot(index=id_col, columns='universe_id', values=y_pred_proba_cols)
            y_pred_proba_df.columns = ['_'.join(map(str, col)) for col in y_pred_proba_df.columns]
            y_pred_proba_df = y_pred_proba_df.reset_index()

        print(f"\ny_pred_proba shape: {y_pred_proba_df.shape}")
        print(f"Columns: {len(y_pred_proba_df.columns)} (1 ID + {len(y_pred_proba_df.columns)-1} prediction columns)")
else:
    print("No prediction files found!")
    y_pred_df = None
    y_pred_proba_df = None

## Explore the Aggregated Data

### y_pred Data

In [ ]:
# Display basic information for y_pred
if y_pred_df is not None:
    print("=== Y_PRED DATA ===")
    print("Columns:", list(y_pred_df.columns[:5]), "..." if len(y_pred_df.columns) > 5 else "")
    print(f"\nNumber of columns: {len(y_pred_df.columns)}")
    print(f"Number of IDs: {len(y_pred_df)}")
    print("\nFirst few rows:")
    display(y_pred_df.head())
else:
    print("No y_pred data available")

In [ ]:
# Display basic information for y_pred_proba
if y_pred_proba_df is not None:
    print("=== Y_PRED_PROBA DATA ===")
    print("Columns:", list(y_pred_proba_df.columns[:5]), "..." if len(y_pred_proba_df.columns) > 5 else "")
    print(f"\nNumber of columns: {len(y_pred_proba_df.columns)}")
    print(f"Number of IDs: {len(y_pred_proba_df)}")
    print("\nFirst few rows:")
    display(y_pred_proba_df.head())
else:
    print("No y_pred_proba data available")

### y_pred_proba Data

In [ ]:
# Check prediction columns for y_pred
if y_pred_df is not None:
    pred_columns = [c for c in y_pred_df.columns if c.startswith('pred_')]
    print("=== Y_PRED ===")
    print(f"Number of prediction columns (universes): {len(pred_columns)}")
    print(f"First 10 prediction columns: {pred_columns[:10]}")

# Check prediction columns for y_pred_proba
if y_pred_proba_df is not None:
    pred_columns = [c for c in y_pred_proba_df.columns if c.startswith('pred_') or 'proba' in c.lower()]
    print("\n=== Y_PRED_PROBA ===")
    print(f"Number of prediction columns (universes): {len(pred_columns)}")
    print(f"First 10 prediction columns: {pred_columns[:10]}")

In [ ]:
# Check for missing predictions (NaN values) in y_pred
if y_pred_df is not None:
    pred_columns = [c for c in y_pred_df.columns if c.startswith('pred_')]
    missing_counts = y_pred_df[pred_columns].isna().sum()
    print("=== Y_PRED Missing Predictions ===")
    if missing_counts.sum() > 0:
        print("Missing predictions per universe:")
        print(missing_counts[missing_counts > 0])
    else:
        print("No missing predictions - all IDs have predictions in all universes!")

# Check for missing predictions (NaN values) in y_pred_proba
if y_pred_proba_df is not None:
    pred_columns = [c for c in y_pred_proba_df.columns if c.startswith('pred_') or 'proba' in c.lower()]
    missing_counts = y_pred_proba_df[pred_columns].isna().sum()
    print("\n=== Y_PRED_PROBA Missing Predictions ===")
    if missing_counts.sum() > 0:
        print("Missing predictions per universe:")
        print(missing_counts[missing_counts > 0])
    else:
        print("No missing predictions - all IDs have predictions in all universes!")

## Save Aggregated Data (Optional)

Save the aggregated predictions to a parquet file.

In [ ]:
# Create output directory if it doesn't exist
output_dir = Path("output/runs") / str(run_id)

# Save y_pred to parquet
if y_pred_df is not None:
    output_file = output_dir / "aggregated_y_pred_test.parquet"
    y_pred_df.to_parquet(output_file, index=False)
    print(f"Saved aggregated y_pred to: {output_file}")
    print(f"File size: {output_file.stat().st_size / 1024 / 1024:.2f} MB")

# Save y_pred_proba to parquet
if y_pred_proba_df is not None:
    output_file = output_dir / "aggregated_y_pred_proba_test.parquet"
    y_pred_proba_df.to_parquet(output_file, index=False)
    print(f"\nSaved aggregated y_pred_proba to: {output_file}")
    print(f"File size: {output_file.stat().st_size / 1024 / 1024:.2f} MB")